### Kelompok 13 | Muhammad Rayhan Mumtaz 2024081040 | Nia Koifa Nia Koifa 2024011001
### Tugas 7

In [55]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import accuracy_score

### 1. Memuat Data (Ambil 15 data pertama dari dataset kuesionermu)

In [56]:
file_path = 'Survei Preferensi Menonton_ Pengembangan Sistem Rekomendasi Film Berbasis AI  (Jawaban) - Form Responses 1.csv'
df_full = pd.read_csv(file_path)
df = df_full.head(15).copy()

### 2. Menyederhanakan Nama Kolom Agar Mudah Dibaca

In [57]:
df.columns = [
    'Timestamp', 'Nama', 'Gender', 'Universitas', 'Angkatan',
    'Frekuensi_Nonton', 'Perangkat', 'Bahasa', 'Genre',
    'Faktor_Rating', 'Faktor_Aktor', 'Faktor_Alur', 'Faktor_Visual', 'Minat'
]

### 2.5 Data Cleaning (Pembersihan Teks Universitas)

In [58]:
# A. Ubah semua teks menjadi huruf kecil (lowercase) dan hapus spasi di awal/akhir
df['Universitas'] = df['Universitas'].str.lower().str.strip()

# B. Daftar variasi penulisan UPJ (termasuk typo seperti "univeesitan")
variasi_upj = [
    'universitas pembangunan jaya', 
    'universitas pembangunan jaya bintaro',
    'univeesitan pembangunan jaya', # Menangani typo dari CSV
    'upj'
]

# C. Timpa semua variasi di atas menjadi satu nama standar: "upj"
df['Universitas'] = df['Universitas'].replace(variasi_upj, 'upj')

print("=== CONTOH HASIL DATA CLEANING (KOLOM UNIVERSITAS) ===")
display(df[['Nama', 'Universitas']].head(15))
print("\n")

=== CONTOH HASIL DATA CLEANING (KOLOM UNIVERSITAS) ===


,Nama,Universitas
0,Rayhan Christian Wewengkang,upj
1,Ruud Zaki Bramdani,upj
2,Fizar Erlansyah,upj
3,Laurensius Jovito Mahaputra Darsono,upj
4,Freyadiva,upj
5,Dea,upj
6,Adam Jagwani Syarif,upj
7,Rexzy Febriano Chasan,upj
8,Fitria Azzahra Nur Syahdiyah,upj
9,Chelsea Meichika,upj


### 3. PREPROCESSING

In [59]:
le_minat = LabelEncoder()
df['Minat'] = le_minat.fit_transform(df['Minat'])

# Mengubah fitur teks menjadi angka numerik
fitur_kategorikal = ['Gender', 'Frekuensi_Nonton', 'Perangkat', 'Bahasa', 'Universitas']
for col in fitur_kategorikal:
    df[col] = LabelEncoder().fit_transform(df[col])

# Menyesuaikan skala faktor (1-5 menjadi 0-4)
faktor_cols = ['Faktor_Rating', 'Faktor_Aktor', 'Faktor_Alur', 'Faktor_Visual']
for col in faktor_cols:
    df[col] = df[col] - 1

# Memisahkan Fitur (X) dan Target (y)
# (Sekarang memasukkan 'Universitas' ke dalam Fitur X)
X = df[['Gender', 'Universitas', 'Frekuensi_Nonton', 'Perangkat', 'Bahasa', 
        'Faktor_Rating', 'Faktor_Aktor', 'Faktor_Alur', 'Faktor_Visual']]
y = df['Minat']

### 4. MEMBAGI DATA LATIH & DATA UJI

In [60]:
X_train, y_train = X.iloc[:5], y.iloc[:5]
X_test, y_test = X.iloc[5:15], y.iloc[5:15]
nama_test = df['Nama'].iloc[5:15].values

# --- SOLUSI ERROR ADA DI SINI ---
# Menghitung jumlah maksimal kategori di setiap kolom untuk memberi tahu 
# model agar tidak kaget jika menemui fitur baru di Data Testing.
batas_kategori = X.max().values + 1

### 5. MELATIH MODEL NAIVE BAYES

In [61]:
model = CategoricalNB(alpha=1.0, min_categories=batas_kategori)
model.fit(X_train, y_train)

CategoricalNB(min_categories=array([2, 1, 3, 3, 3, 5, 5, 5, 5]))

### 6. MELAKUKAN PREDIKSI PADA 10 DATA TESTING

In [62]:
y_pred = model.predict(X_test)
prediksi_teks = le_minat.inverse_transform(y_pred)
aktual_teks = le_minat.inverse_transform(y_test)

### 7. MENGHITUNG EVALUASI (Aktivitas 2)

In [63]:
# 7. MENGHITUNG EVALUASI (Aktivitas 2)
jumlah_benar = sum(prediksi_teks == aktual_teks)
total_data = len(y_test)
akurasi = accuracy_score(y_test, y_pred) * 100

print("=====================================================")
print("        HASIL PREDIKSI PADA 10 DATA TESTING          ")
print("=====================================================")
for i in range(total_data):
    status = "BENAR" if prediksi_teks[i] == aktual_teks[i] else "SALAH"
    print(f"{i+1:2d}. {nama_test[i][:20]:<20} | Prediksi: {prediksi_teks[i][:15]:<15} | Aktual: {aktual_teks[i][:15]:<15} -> {status}")

print("\n=====================================================")
print("     JAWABAN AKTIVITAS 2: KESIMPULAN EVALUASI        ")
print("=====================================================")

# 1. Menghitung jumlah prediksi benar
print(f"1. Jumlah Prediksi Benar : {jumlah_benar} dari {total_data} data testing")

# 2. Menghitung akurasi
print(f"2. Akurasi Model         : {akurasi:.2f}%")

# 3. Menginterpretasikan hasilnya
print("3. Interpretasi Hasil    :")
if akurasi >= 70:
    print("   Model berkinerja BAIK.\n   AI sukses memprediksi data baru berdasarkan pola dari data latih.")
else:
    print("   Model berkinerja KURANG MAKSIMAL.\n   Hal ini wajar karena 5 Data Latih belum cukup bagi AI untuk belajar secara optimal.")
print("=====================================================")

        HASIL PREDIKSI PADA 10 DATA TESTING          
 1. Dea                  | Prediksi: Ya (Tertarik)   | Aktual: Tidak (Tidak Te -> SALAH
 2. Adam Jagwani Syarif  | Prediksi: Ya (Tertarik)   | Aktual: Tidak (Tidak Te -> SALAH
 3. Rexzy Febriano Chasa | Prediksi: Ya (Tertarik)   | Aktual: Ya (Tertarik)   -> BENAR
 4. Fitria Azzahra Nur S | Prediksi: Ya (Tertarik)   | Aktual: Tidak (Tidak Te -> SALAH
 5. Chelsea Meichika     | Prediksi: Ya (Tertarik)   | Aktual: Tidak (Tidak Te -> SALAH
 6. Syenni Neisya Aureli | Prediksi: Ya (Tertarik)   | Aktual: Ya (Tertarik)   -> BENAR
 7. Kaila Zanita         | Prediksi: Ya (Tertarik)   | Aktual: Tidak (Tidak Te -> SALAH
 8. Zidane Tirta Nugraha | Prediksi: Ya (Tertarik)   | Aktual: Tidak (Tidak Te -> SALAH
 9. Mutiara Nabila       | Prediksi: Ya (Tertarik)   | Aktual: Ya (Tertarik)   -> BENAR
10. Diandra Cynthia Ashr | Prediksi: Ya (Tertarik)   | Aktual: Ya (Tertarik)   -> BENAR

     JAWABAN AKTIVITAS 2: KESIMPULAN EVALUASI        
1. Jumlah P